# NLP #1: What is NLP? The NLP Pipeline

**AI Teacher** · Interview prep notebook for the video *What is NLP? The NLP Pipeline* (taught by Gonu Jha Ji with Priya, Rahul and Sneha).

Run the cells from top to bottom (`Shift + Enter`). Every output you see was produced by running this notebook. Change the examples and run them again - that is the best way to learn.

## What you will learn
1. What NLP, NLU and NLG mean (with examples you use every day)
2. Why human language is hard for computers
3. The 7 steps of an NLP pipeline - and a **complete working pipeline** in 20 lines
4. Rule-based vs machine-learning NLP
5. The 4 interview questions from the video, with model answers

In [1]:
# One-time setup (uncomment if a library is missing):
# !pip install nltk scikit-learn pandas
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
print("Ready!")

Ready!


## 1. Key definitions

| Term | Meaning | Example |
|---|---|---|
| **NLP** (Natural Language Processing) | The field of AI that lets computers understand, process and generate human language | Gmail spam filter, Google Translate, ChatGPT |
| **NLU** (Natural Language Understanding) | Text in, *meaning* out | "Book a cab to the airport" → intent = `book_cab`, place = `airport` |
| **NLG** (Natural Language Generation) | Meaning in, *text* out | `{eta: 5}` → "Your cab will arrive in 5 minutes." |
| **Corpus** | A collection of documents (texts) used for training or analysis | 10,000 Zomato reviews |
| **Token** | One unit of text after splitting - usually a word or sub-word | "good", "movie" |
| **Pipeline** | The sequence of steps that turns raw text into a working model | collect → clean → ... → deploy |

> **Remember the parrot from Gonu Jha's story:** repeating words is not understanding. NLP is about *meaning*, not just words.

## 2. NLU by hand: understanding a cab booking

A chatbot must find the **intent** (what the user wants) and the **entities / slots** (the details).
Real systems use ML models, but a tiny rule-based version shows the idea clearly.

In [2]:
import re

def understand(sentence):
    """A tiny rule-based NLU: intent + slots."""
    s = sentence.lower()
    intent = "book_cab" if re.search(r"\b(book|get|need)\b.*\b(cab|taxi|ride)\b", s) else "unknown"
    slots = {}
    if m := re.search(r"from ([a-z ]+?) to", s):
        slots["from"] = m.group(1).strip()
    if m := re.search(r"to (?:the )?([a-z]+)", s):
        slots["to"] = m.group(1)
    if m := re.search(r"(\d{1,2}) ?(am|pm)", s):
        slots["time"] = f"{m.group(1)} {m.group(2).upper()}"
    if "tomorrow" in s:
        slots["day"] = "tomorrow"
    return intent, slots

print(understand("Book a cab from Andheri to the airport at 6 AM tomorrow"))
print(understand("What is the weather today?"))

('book_cab', {'from': 'andheri', 'to': 'airport', 'time': '6 AM', 'day': 'tomorrow'})
('unknown', {})


## 3. NLG by hand: writing the reply
NLG is the reverse direction: structured data in, a natural sentence out. The simplest NLG is a **template**;
ChatGPT-style models generate the sentence word by word instead.

In [3]:
def reply(intent, slots):
    if intent != "book_cab":
        return "Sorry, I can only book cabs right now."
    return (f"Your cab from {slots['from'].title()} to the {slots['to']} "
            f"is booked for {slots['time']} {slots.get('day', 'today')}.")

print(reply(*understand("Book a cab from Andheri to the airport at 6 AM tomorrow")))

Your cab from Andheri to the airport is booked for 6 AM tomorrow.


## 4. Why is NLP hard?

| Problem | Example | Why it breaks simple rules |
|---|---|---|
| **Ambiguity** | "I saw her duck" | the bird, or she bent down? |
| **Sarcasm** | "Wow, great service!" after a 2-hour delay | positive words, negative meaning |
| **Negation** | "The food was not good" | "good" is positive, the sentence is negative |
| **Code-mixing** | "Movie mast thi but ending boring" | Hindi + English in one sentence |
| **Spelling / slang** | "gr8", "sooo good" | words not in any dictionary |

Let us *see* these failures with a naive word-counting sentiment classifier:

In [4]:
POSITIVE = {"good", "great", "love", "loved", "amazing", "wow", "best"}
NEGATIVE = {"bad", "worst", "boring", "late", "hate", "poor"}

def naive_sentiment(text):
    words = re.findall(r"[a-z]+", text.lower())
    score = sum(w in POSITIVE for w in words) - sum(w in NEGATIVE for w in words)
    return "positive" if score > 0 else "negative" if score < 0 else "neutral"

tests = {
    "The food was amazing": "positive",
    "Wow, great service! Only a 2 hour delay": "negative (sarcasm)",
    "The food was not good": "negative (negation)",
    "Movie mast thi but ending boring": "mixed (code-mixed)",
    "Delivery was gr8": "positive (slang)",
}
for text, truth in tests.items():
    print(f"{naive_sentiment(text):9s} <- model | truth -> {truth:20s} | {text}")

positive  <- model | truth -> positive             | The food was amazing
positive  <- model | truth -> negative (sarcasm)   | Wow, great service! Only a 2 hour delay
positive  <- model | truth -> negative (negation)  | The food was not good
negative  <- model | truth -> mixed (code-mixed)   | Movie mast thi but ending boring
neutral   <- model | truth -> positive (slang)     | Delivery was gr8


Only the first sentence is handled correctly. This is exactly why we need proper preprocessing and machine learning - the topics of the next videos.

## 5. The NLP pipeline (7 steps)

| # | Step | What happens | Video |
|---|---|---|---|
| 1 | Data collection | Gather reviews, chats, tickets... | |
| 2 | Text cleaning | Remove HTML, URLs, noise | NLP #2 |
| 3 | Preprocessing | Lowercase, tokenize, stopwords, lemmatize | NLP #2, #3, #4 |
| 4 | Feature extraction | Turn text into numbers: Bag of Words, TF-IDF, embeddings | NLP #5, #6 |
| 5 | Modeling | Train a classifier (Naive Bayes, Logistic Regression, BERT...) | |
| 6 | Evaluation | Accuracy, precision, recall, **F1** on unseen data | |
| 7 | Deployment | Serve the model in an app and monitor it | |

The code from the video (steps 2-3):

In [5]:
text = "I LOVED this movie!!! 10/10, would watch again :)"
text = text.lower()                      # Step 1: lowercase
clean = re.sub(r"[^a-z\s]", "", text)   # Step 2: keep only letters and spaces
tokens = clean.split()                   # Step 3: tokenization
print(tokens)

['i', 'loved', 'this', 'movie', 'would', 'watch', 'again']


## 6. A complete pipeline in ~20 lines
Below, all 7 steps run on a tiny dataset of restaurant reviews: clean → features (Bag of Words) →
model (Naive Bayes) → evaluation (F1) → "deployment" (a `predict` function). Real projects use thousands of
reviews, but the steps are exactly the same.

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

# 1. Data collection
train = [("food was amazing and fresh", 1), ("loved the biryani", 1), ("great taste quick delivery", 1),
         ("best paneer in town", 1), ("delivery was late and food cold", 0), ("worst service ever", 0),
         ("food was bad and stale", 0), ("very poor packaging", 0)]
test = [("amazing biryani and quick delivery", 1), ("cold food and late delivery", 0),
        ("fresh and tasty paneer", 1), ("bad taste poor service", 0)]

# 2-3. Cleaning + preprocessing
def clean(text):
    return re.sub(r"[^a-z\s]", " ", text.lower())

# 4. Features: Bag of Words (fit on TRAIN only)
vec = CountVectorizer(preprocessor=clean)
X_train = vec.fit_transform([t for t, _ in train])
X_test = vec.transform([t for t, _ in test])

# 5. Model
model = MultinomialNB().fit(X_train, [y for _, y in train])

# 6. Evaluation
pred = model.predict(X_test)
print("F1 on unseen reviews:", f1_score([y for _, y in test], pred))

# 7. "Deployment"
def predict(review):
    return "positive" if model.predict(vec.transform([review]))[0] == 1 else "negative"

print(predict("the paneer was amazing"))
print(predict("late delivery and stale food"))

F1 on unseen reviews: 1.0
positive
negative


## 7. Rule-based vs ML-based NLP

| | Rule-based (regex, keyword lists) | ML-based (trained models) |
|---|---|---|
| Best for | Small, fixed patterns: phone numbers, PAN, dates | Messy, varied language: reviews, chats |
| Training data | Not needed | Needed (labelled examples) |
| Explainability | Very easy | Harder |
| Handles new phrasing | No | Yes (generalizes) |

Real systems mix both. Example: rules are perfect for extracting an Indian mobile number or a PAN number:

In [7]:
msg = "Call me on 9876543210 or 91234 56789. My PAN is ABCDE1234F."
phones = re.findall(r"\b[6-9]\d{4}\s?\d{5}\b", msg)
pan = re.findall(r"\b[A-Z]{5}\d{4}[A-Z]\b", msg)
print("Phones:", phones)
print("PAN:", pan)

Phones: ['9876543210', '91234 56789']
PAN: ['ABCDE1234F']


## 8. How NLP evolved
1. **Rule-based** (1950s-90s): regex and hand-written grammar rules
2. **Classical ML** (2000s): Naive Bayes, SVM on Bag of Words / TF-IDF features
3. **Deep learning** (2013+): Word2Vec, RNN, LSTM
4. **Transformers & LLMs** (2018+): BERT, GPT - today's GenAI

## 9. Interview questions (from the video)

**Q1. What is NLP? Explain with an example.**
NLP is the field of AI that lets computers understand and generate human language. It converts unstructured
text into numbers a model can learn from. *Example:* classifying Zomato reviews as positive or negative.
Always name one concrete task: classification, translation, NER, question answering, or generation.

**Q2. Walk me through an NLP pipeline.**
Collect data → clean noise → preprocess (lowercase, tokenize, stopwords, lemmatize) → extract features
(BoW / TF-IDF / embeddings) → train → evaluate with F1 → deploy and monitor.
*Bonus point:* say which steps you skip for BERT or LLMs (heavy cleaning, stopword removal, stemming).

**Q3. Difference between NLP, NLU and NLG?**
NLP is the whole field. NLU = understanding (intent and entity detection). NLG = producing text (summaries,
chatbot replies). A chatbot = NLU (understand the query) + NLG (write the reply).

**Q4. Rule-based or ML-based NLP: when to use which?**
Rules for small, fixed patterns with no training data (phone numbers, PAN, dates) - easy to explain.
ML for messy, varied language. Real systems mix both: rules for clean-up, ML for meaning.

### Common mistakes
- Defining NLP without a real-world example
- Forgetting evaluation and deployment in the pipeline
- Saying the same preprocessing works for every model
- Ignoring code-mixed Hinglish text in Indian use cases

## 10. Practice
**Task:** list 3 NLP features in an app on your phone and name the NLP task behind each.
Then try this coding exercise: extend `understand()` so it also detects the intent `cancel_cab`
("Cancel my cab please").

In [8]:
# Solution
def understand_v2(sentence):
    if re.search(r"\bcancel\b.*\b(cab|taxi|ride)\b", sentence.lower()):
        return "cancel_cab", {}
    return understand(sentence)

print(understand_v2("Cancel my cab please"))
print(understand_v2("Book a cab from Powai to the airport at 9 PM"))

('cancel_cab', {})
('book_cab', {'from': 'powai', 'to': 'airport', 'time': '9 PM'})


**Next video:** NLP #2 - Text Preprocessing.